# Gold Regimes, Hidden Seasonality, and Next-Day Forecasting
### A publication-ready Kaggle notebook for extracting signal from 25 years of gold price history

Gold is not just another asset. It sits at the intersection of inflation fear, macro uncertainty, and risk sentiment, which makes its price path unusually rich for time-series analysis.

This notebook is built for readers who want **signal over clutter**. Instead of dumping generic charts, we will:
- uncover a few non-obvious patterns in long-run gold behavior,
- segment the market into interpretable regimes,
- build a clean forecasting baseline,
- add one meaningful model upgrade,
- explain what the model is actually learning.

If you only read one public notebook on this dataset, the goal is simple: it should leave you with reusable ideas, not just screenshots.


## Introduction

### Why this dataset matters
Gold is one of the clearest macro-sensitive instruments in public markets. A long OHLCV history lets us study trend persistence, volatility clustering, drawdowns, and short-horizon predictability in one place.

### What you will learn
1. How to turn raw OHLCV data into a clean, analysis-ready time-series frame.
2. Which EDA views actually reveal structure in gold instead of repeating standard plots.
3. How to engineer practical forecasting features without leaking future information.
4. Why a simple linear baseline is useful, and where a nonlinear model adds value.
5. How to interpret model behavior with permutation importance and error slices.

### Notebook roadmap
1. Setup
2. Data loading
3. Quick overview
4. Data cleaning
5. High-signal EDA
6. Key insights summary
7. Feature engineering
8. Baseline model
9. Improved model
10. Validation
11. Model comparison
12. Explainability
13. Final insights, pitfalls, and next steps


## Setup

The notebook is intentionally lightweight:
- Kaggle-friendly file discovery using `/kaggle/input`
- deterministic seeds for reproducibility
- defensive parsing for date and numeric columns
- models that run quickly on CPU without memory issues


In [ ]:
# 1) Setup
import os
import gc
import glob
import random
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.dpi'] = 120

PALETTE = ['#b58900', '#005f73', '#ae2012', '#3a5a40', '#6c757d']

print(f'Seed fixed at {SEED}')


## Data Loading

This notebook auto-detects the main CSV from Kaggle input folders. If you are previewing it locally, it also supports a fallback local file.


In [ ]:
# 2) Data loading
INPUT_ROOT = '/kaggle/input'
LOCAL_FALLBACK = './finalgolddata.csv'

csv_paths = glob.glob(os.path.join(INPUT_ROOT, '**', '*.csv'), recursive=True)
if not csv_paths and os.path.exists(LOCAL_FALLBACK):
    csv_paths = [LOCAL_FALLBACK]
if not csv_paths:
    raise FileNotFoundError('No CSV file found under /kaggle/input and local fallback is missing.')

def pick_main_csv(paths):
    gold_first = [p for p in paths if 'gold' in os.path.basename(p).lower()]
    ranked = gold_first if gold_first else paths
    ranked = sorted(ranked, key=lambda p: os.path.getsize(p), reverse=True)
    return ranked[0]

data_path = pick_main_csv(csv_paths)
raw = pd.read_csv(data_path)

print('Selected file:', data_path)
print('Raw shape:', raw.shape)
display(raw.head())


## Quick Overview

Before doing anything clever, we answer the only questions that matter first:
- Is the series chronologically usable?
- Are the fields consistent with OHLCV structure?
- Is there obvious missingness, duplication, or suspicious volume behavior?


In [ ]:
# 3) Quick overview
summary = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing_values': raw.isna().sum(),
    'missing_pct': (raw.isna().mean() * 100).round(2),
    'n_unique': raw.nunique()
})

print('Columns:', list(raw.columns))
print('Duplicate rows:', raw.duplicated().sum())
display(summary)


## Data Cleaning

We keep cleaning conservative and explicit:
- normalize column names,
- parse the date safely,
- coerce numeric fields,
- sort chronologically,
- preserve zero volume as a real signal instead of silently dropping it,
- create a few structural columns that help both EDA and modeling.


In [ ]:
# 4) Data cleaning

df = raw.copy()
df.columns = [c.strip().replace(' ', '_').replace('-', '_') for c in df.columns]

# Identify the date column defensively
candidate_dates = [c for c in df.columns if any(k in c.lower() for k in ['date', 'time', 'timestamp'])]
if not candidate_dates:
    raise ValueError('No date-like column found.')
date_col = candidate_dates[0]

df[date_col] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)

rename_map = {}
for c in df.columns:
    low = c.lower()
    if low == 'open':
        rename_map[c] = 'Open'
    elif low == 'high':
        rename_map[c] = 'High'
    elif low == 'low':
        rename_map[c] = 'Low'
    elif low == 'close':
        rename_map[c] = 'Close'
    elif 'vol' in low:
        rename_map[c] = 'Volume'
    elif c == date_col:
        rename_map[c] = 'Date'

df = df.rename(columns=rename_map)
required_cols = ['Date', 'Open', 'High', 'Low', 'Close']
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing required OHLC columns: {missing_required}')
if 'Volume' not in df.columns:
    df['Volume'] = 0

for c in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna(subset=['Date']).sort_values('Date').drop_duplicates(subset='Date', keep='last').reset_index(drop=True)

# Conservative fill for rare numeric gaps after coercion
price_cols = ['Open', 'High', 'Low', 'Close']
df[price_cols] = df[price_cols].ffill().bfill()
df['Volume'] = df['Volume'].fillna(0)

# Structural EDA fields
price_range = (df['High'] - df['Low']).replace(0, np.nan)
df['Daily_Return'] = df['Close'].pct_change()
df['Log_Return'] = np.log1p(df['Daily_Return'])
df['Range_Pct'] = (df['High'] - df['Low']) / df['Close'].replace(0, np.nan)
df['Body_Pct'] = (df['Close'] - df['Open']) / df['Open'].replace(0, np.nan)
df['Close_Location'] = ((df['Close'] - df['Low']) / price_range).clip(0, 1)
df['Volume_Zero_Flag'] = (df['Volume'] == 0).astype(int)
df['Log_Volume'] = np.log1p(df['Volume'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Weekday'] = df['Date'].dt.day_name().str[:3]

df['Rolling_Vol_21'] = df['Daily_Return'].rolling(21).std() * np.sqrt(252)
df['Momentum_126'] = df['Close'].pct_change(126)
df['Drawdown'] = df['Close'] / df['Close'].cummax() - 1

df['Next_Close'] = df['Close'].shift(-1)
df['Future_5D_Return'] = df['Close'].shift(-5) / df['Close'] - 1

zero_volume_share = df['Volume_Zero_Flag'].mean() * 100
print('Clean shape:', df.shape)
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print(f'Zero-volume rows: {zero_volume_share:.2f}%')
display(df.head())


## EDA

The goal here is not to show everything. It is to isolate the views that actually change our understanding of the series.


In [ ]:
# 5) EDA part 1: long-run structure, drawdowns, and volatility clustering
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

axes[0].plot(df['Date'], df['Close'], color=PALETTE[0], linewidth=2)
axes[0].set_title('Gold close price over time')
axes[0].set_ylabel('Close')

axes[1].fill_between(df['Date'], df['Drawdown'], 0, color=PALETTE[2], alpha=0.35)
axes[1].set_title('Drawdown from running peak')
axes[1].set_ylabel('Drawdown')

axes[2].plot(df['Date'], df['Rolling_Vol_21'], color=PALETTE[1], linewidth=1.8)
axes[2].set_title('21-day annualized realized volatility')
axes[2].set_ylabel('Volatility')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

cagr = (df['Close'].iloc[-1] / df['Close'].iloc[0]) ** (252 / max(len(df), 1)) - 1
max_drawdown = df['Drawdown'].min()
vol_median = df['Rolling_Vol_21'].median()

print(f'Price range: {df["Close"].min():,.1f} to {df["Close"].max():,.1f}')
print(f'Max drawdown: {max_drawdown:.2%}')
print(f'Median 21-day realized volatility: {vol_median:.2%}')


### Interpretation

Three things stand out immediately:
- gold is a strong long-run trend asset, but the path is clearly regime-dependent rather than smooth,
- the deepest drawdowns are clustered instead of evenly distributed,
- volatility is sticky: once it expands, it often stays elevated for a while.

That last point matters because models trained on calm periods usually degrade when the regime shifts.


In [ ]:
# 6) EDA part 2: monthly return heatmap and seasonality
monthly_close = df.set_index('Date')['Close'].resample('M').last()
monthly_return = monthly_close.pct_change().rename('Monthly_Return')
monthly_table = monthly_return.to_frame()
monthly_table['Year'] = monthly_table.index.year
monthly_table['Month'] = monthly_table.index.strftime('%b')
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
heatmap_data = monthly_table.pivot_table(index='Year', columns='Month', values='Monthly_Return')
heatmap_data = heatmap_data.reindex(columns=month_order)

avg_monthly = monthly_table.groupby('Month')['Monthly_Return'].mean().reindex(month_order)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [2.3, 1]})
sns.heatmap(heatmap_data * 100, cmap='RdYlGn', center=0, linewidths=0.3, ax=axes[0], cbar_kws={'label': 'Monthly return (%)'})
axes[0].set_title('Monthly return heatmap by year')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Year')

sns.barplot(x=avg_monthly.values * 100, y=avg_monthly.index, color=PALETTE[0], ax=axes[1])
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Average monthly return')
axes[1].set_xlabel('Return (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

best_month = avg_monthly.idxmax()
worst_month = avg_monthly.idxmin()
print(f'Strongest average month: {best_month} ({avg_monthly.max():.2%})')
print(f'Weakest average month: {worst_month} ({avg_monthly.min():.2%})')


### Interpretation

This is the first real aha moment. Gold does not behave like a featureless trend line. Seasonality is not huge, but it is persistent enough to matter when we build calendar features.

In many financial notebooks, seasonality is either overstated or ignored. Here, the useful stance is somewhere in the middle: treat it as a weak but reusable edge, not a magic rule.


In [ ]:
# 7) EDA part 3: regime map and segment analysis
regime_frame = df[['Date', 'Momentum_126', 'Rolling_Vol_21', 'Future_5D_Return', 'Range_Pct', 'Close_Location']].dropna().copy()
vol_cut = regime_frame['Rolling_Vol_21'].median()
regime_frame['Regime'] = np.select(
    [
        (regime_frame['Momentum_126'] >= 0) & (regime_frame['Rolling_Vol_21'] < vol_cut),
        (regime_frame['Momentum_126'] >= 0) & (regime_frame['Rolling_Vol_21'] >= vol_cut),
        (regime_frame['Momentum_126'] < 0) & (regime_frame['Rolling_Vol_21'] < vol_cut),
        (regime_frame['Momentum_126'] < 0) & (regime_frame['Rolling_Vol_21'] >= vol_cut),
    ],
    ['Uptrend / Low vol', 'Uptrend / High vol', 'Downtrend / Low vol', 'Downtrend / High vol'],
    default='Other'
)

regime_order = ['Uptrend / Low vol', 'Uptrend / High vol', 'Downtrend / Low vol', 'Downtrend / High vol']
regime_stats = regime_frame.groupby('Regime').agg(
    days=('Date', 'size'),
    median_5d_return=('Future_5D_Return', 'median'),
    mean_range_pct=('Range_Pct', 'mean'),
    mean_close_location=('Close_Location', 'mean')
).reindex(regime_order)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.scatterplot(
    data=regime_frame.sample(min(2500, len(regime_frame)), random_state=SEED),
    x='Momentum_126', y='Rolling_Vol_21', hue='Regime', palette='Set2', alpha=0.75, ax=axes[0]
)
axes[0].axvline(0, color='black', linewidth=1, linestyle='--')
axes[0].axhline(vol_cut, color='black', linewidth=1, linestyle='--')
axes[0].set_title('Market regime map: momentum vs volatility')
axes[0].set_xlabel('126-day momentum')
axes[0].set_ylabel('21-day realized volatility')

sns.boxplot(data=regime_frame, x='Regime', y='Future_5D_Return', order=regime_order, palette='Set2', ax=axes[1], showfliers=False)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Forward 5-day return by regime')
axes[1].set_xlabel('')
axes[1].set_ylabel('Future 5-day return')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

display((regime_stats * [1, 100, 100, 100]).rename(columns={
    'median_5d_return': 'median_5d_return_%',
    'mean_range_pct': 'mean_range_%',
    'mean_close_location': 'mean_close_location_%'
}).round(2))


### Interpretation

This is the second aha moment: **high volatility is not automatically bearish**.

Gold spends a large amount of time in an `uptrend / high vol` state, and that regime often behaves very differently from `downtrend / high vol`. That is exactly why regime-aware features can outperform a plain linear model even when both use the same raw OHLCV columns.


In [ ]:
# 8) EDA part 4: an uncommon but useful view of intraday structure
intraday = df[['Date', 'Range_Pct', 'Close_Location', 'Weekday', 'Rolling_Vol_21']].dropna().copy()
intraday['Vol_Bucket'] = pd.qcut(intraday['Rolling_Vol_21'], 4, labels=['Q1 Calm', 'Q2', 'Q3', 'Q4 Turbulent'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.violinplot(data=intraday, x='Vol_Bucket', y='Close_Location', palette='crest', ax=axes[0], cut=0)
axes[0].set_title('Where does gold close within the daily range?')
axes[0].set_xlabel('Volatility bucket')
axes[0].set_ylabel('Close location in daily range')

weekday_range = intraday.groupby('Weekday')['Range_Pct'].mean().reindex(['Mon', 'Tue', 'Wed', 'Thu', 'Fri'])
sns.barplot(x=weekday_range.index, y=weekday_range.values * 100, color=PALETTE[1], ax=axes[1])
axes[1].set_title('Average intraday range by weekday')
axes[1].set_xlabel('Weekday')
axes[1].set_ylabel('Average range (%)')

plt.tight_layout()
plt.show()


### Interpretation

`Close_Location` is a small but surprisingly informative feature. It tells us whether the market tends to finish near the highs or lows of the day, which is often more useful than looking at `Open` and `Close` alone.

This is also a good example of notebook originality: a compact structural view can say more than another generic histogram.


## Key Insights Summary

We can now compress the EDA into a small set of takeaways that are actually worth remembering.


In [ ]:
# 9) Key insights summary
up_days = (df['Daily_Return'] > 0).sum()
down_days = (df['Daily_Return'] < 0).sum()
flat_days = (df['Daily_Return'] == 0).sum()

regime_best = regime_stats['median_5d_return'].idxmax()
regime_worst = regime_stats['median_5d_return'].idxmin()

insights = f"""
### Mini checklist of high-signal findings
- The dataset covers **{df['Date'].min().date()} to {df['Date'].max().date()}** with **{len(df):,} trading rows**.
- Gold moved from **{df['Close'].min():,.1f}** to **{df['Close'].max():,.1f}**, but with a maximum drawdown of **{df['Drawdown'].min():.1%}**.
- About **{df['Volume_Zero_Flag'].mean():.1%}** of rows have zero volume, so volume needs careful treatment instead of blind trust.
- Average monthly strength is highest in **{best_month}** and weakest in **{worst_month}**.
- The most supportive 5-day regime is **{regime_best}**, while the weakest is **{regime_worst}**.
- Up days: **{up_days:,}** | Down days: **{down_days:,}** | Flat days: **{flat_days:,}**.
"""

display(Markdown(insights))


## Feature Engineering

For forecasting, the goal is simple: use only information available at time `t` to predict the next session's close.

We combine four feature families:
- lagged price levels and returns,
- rolling statistics and trend measures,
- volume transformations,
- calendar and regime-aware signals.


In [ ]:
# 10) Feature engineering
model_df = df.copy()

# Lag features
for lag in [1, 2, 3, 5, 10, 21, 63]:
    model_df[f'close_lag_{lag}'] = model_df['Close'].shift(lag)
    model_df[f'return_lag_{lag}'] = model_df['Daily_Return'].shift(lag)
    model_df[f'volume_lag_{lag}'] = model_df['Log_Volume'].shift(lag)

# Rolling features
for window in [5, 10, 21, 63]:
    model_df[f'close_ma_{window}'] = model_df['Close'].rolling(window).mean()
    model_df[f'close_std_{window}'] = model_df['Close'].rolling(window).std()
    model_df[f'return_mean_{window}'] = model_df['Daily_Return'].rolling(window).mean()
    model_df[f'return_std_{window}'] = model_df['Daily_Return'].rolling(window).std()
    model_df[f'range_mean_{window}'] = model_df['Range_Pct'].rolling(window).mean()

model_df['momentum_21'] = model_df['Close'].pct_change(21)
model_df['momentum_63'] = model_df['Close'].pct_change(63)
model_df['momentum_126'] = model_df['Momentum_126']
model_df['ma_ratio_5_21'] = model_df['close_ma_5'] / model_df['close_ma_21'] - 1
model_df['ma_ratio_10_63'] = model_df['close_ma_10'] / model_df['close_ma_63'] - 1
model_df['vol_ratio_5_21'] = model_df['return_std_5'] / model_df['return_std_21']
model_df['volume_ratio_5_21'] = model_df['Log_Volume'].rolling(5).mean() / model_df['Log_Volume'].rolling(21).mean()

# Calendar features
model_df['day_of_week'] = model_df['Date'].dt.dayofweek
model_df['month'] = model_df['Date'].dt.month
model_df['quarter'] = model_df['Date'].dt.quarter
model_df['week_of_year'] = model_df['Date'].dt.isocalendar().week.astype(int)

# Target
model_df['target_next_close'] = model_df['Next_Close']

feature_cols = [
    c for c in model_df.columns
    if c not in ['Date', 'Month_Name', 'Weekday', 'Next_Close', 'Future_5D_Return', 'target_next_close']
]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(model_df[c])]

model_df = model_df.dropna(subset=['target_next_close']).reset_index(drop=True)
ready = model_df.dropna().reset_index(drop=True)

X = ready[feature_cols]
y = ready['target_next_close']

holdout_size = max(int(len(ready) * 0.15), 250)
split_idx = len(ready) - holdout_size
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_train, dates_test = ready['Date'].iloc[:split_idx], ready['Date'].iloc[split_idx:]

print('Modeling rows:', len(ready))
print('Number of features:', len(feature_cols))
print('Train / holdout:', X_train.shape, X_test.shape)


## Baseline Model

A baseline should be easy to trust. We start with a scaled Ridge regression:
- fast,
- stable,
- interpretable,
- hard to accidentally overfit on this dataset size.

If a more complex model cannot beat this clean baseline, it does not deserve to stay.


In [ ]:
# 11) Baseline model

def safe_mape(y_true, y_pred):
    denom = np.maximum(np.abs(y_true), 1e-8)
    return np.mean(np.abs((y_true - y_pred) / denom))


def evaluate_model_cv(model, X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    oof_pred = pd.Series(index=y.index, dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)
        oof_pred.iloc[va_idx] = pred

        rows.append({
            'fold': fold,
            'MAE': mean_absolute_error(y_va, pred),
            'RMSE': np.sqrt(mean_squared_error(y_va, pred)),
            'MAPE': safe_mape(y_va.values, pred),
            'R2': r2_score(y_va, pred)
        })

    result = pd.DataFrame(rows)
    return result, oof_pred

baseline_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=2.0))
])

baseline_cv, baseline_oof = evaluate_model_cv(baseline_model, X_train, y_train)
display(baseline_cv.round(4))
display(baseline_cv.mean(numeric_only=True).to_frame('Baseline mean').T.round(4))


## Improved Model

Now we add one upgrade that is meaningful rather than flashy: a histogram gradient boosting regressor.

Why this is a sensible next step:
- it captures nonlinear interactions between lagged trend and volatility,
- it handles mixed feature scales well,
- it stays lightweight enough for Kaggle CPU runs.


In [ ]:
# 12) Improved model
improved_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=6,
        max_iter=350,
        min_samples_leaf=20,
        l2_regularization=0.1,
        random_state=SEED
    ))
])

improved_cv, improved_oof = evaluate_model_cv(improved_model, X_train, y_train)
display(improved_cv.round(4))
display(improved_cv.mean(numeric_only=True).to_frame('Improved mean').T.round(4))


## Validation

Cross-validation is useful, but a notebook becomes more credible when it also shows a clean final holdout test.

We reserve the last 15% of the timeline as an untouched out-of-sample window and evaluate both models there.


In [ ]:
# 13) Final validation on holdout
baseline_model.fit(X_train, y_train)
improved_model.fit(X_train, y_train)

pred_baseline = baseline_model.predict(X_test)
pred_improved = improved_model.predict(X_test)

results = []
for name, pred in [('Ridge baseline', pred_baseline), ('HGB improved', pred_improved)]:
    results.append({
        'Model': name,
        'Holdout MAE': mean_absolute_error(y_test, pred),
        'Holdout RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'Holdout MAPE': safe_mape(y_test.values, pred),
        'Holdout R2': r2_score(y_test, pred)
    })

holdout_table = pd.DataFrame(results).sort_values('Holdout MAE').reset_index(drop=True)
display(holdout_table.round(4))

plot_df = pd.DataFrame({
    'Date': dates_test,
    'Actual': y_test.values,
    'Ridge baseline': pred_baseline,
    'HGB improved': pred_improved
})

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(plot_df['Date'], plot_df['Actual'], label='Actual', color='black', linewidth=2)
ax.plot(plot_df['Date'], plot_df['Ridge baseline'], label='Ridge baseline', color=PALETTE[1], alpha=0.8)
ax.plot(plot_df['Date'], plot_df['HGB improved'], label='HGB improved', color=PALETTE[0], alpha=0.9)
ax.set_title('Holdout window: actual vs predicted next-day close')
ax.set_ylabel('Next-day close')
ax.legend()
plt.tight_layout()
plt.show()


## Model Comparison

A public notebook should make model tradeoffs obvious. Here is the compact comparison table a reader can scan in seconds.


In [ ]:
# 14) Model comparison table
comparison = pd.DataFrame([
    {
        'Model': 'Ridge baseline',
        'CV MAE': baseline_cv['MAE'].mean(),
        'CV RMSE': baseline_cv['RMSE'].mean(),
        'CV MAPE': baseline_cv['MAPE'].mean(),
        'Holdout MAE': holdout_table.loc[holdout_table['Model'] == 'Ridge baseline', 'Holdout MAE'].iloc[0],
        'Notes': 'Strong linear benchmark with scaled features'
    },
    {
        'Model': 'HGB improved',
        'CV MAE': improved_cv['MAE'].mean(),
        'CV RMSE': improved_cv['RMSE'].mean(),
        'CV MAPE': improved_cv['MAPE'].mean(),
        'Holdout MAE': holdout_table.loc[holdout_table['Model'] == 'HGB improved', 'Holdout MAE'].iloc[0],
        'Notes': 'Captures nonlinear interactions and regime effects'
    }
]).sort_values('Holdout MAE').reset_index(drop=True)

display(comparison.round(4))


## Explainability

Accuracy without interpretation is not enough for financial time series.

We use permutation importance on the holdout window to answer a simple question: **which features does the improved model actually depend on most?**


In [ ]:
# 15) Explainability with permutation importance
perm = permutation_importance(
    improved_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=SEED,
    scoring='neg_mean_absolute_error'
)

perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=perm_df, x='importance_mean', y='feature', color=PALETTE[0], ax=ax)
ax.set_title('Top permutation importances for the improved model')
ax.set_xlabel('Importance (drop in score when shuffled)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

top_features = ', '.join(perm_df['feature'].head(5).tolist())
print('Top 5 features:', top_features)


### Interpretation

If the model leans heavily on rolling means, lagged closes, momentum, and volatility ratios, that is a healthy sign. Those are economically plausible drivers for short-horizon price forecasting.

If it were dominated by noisy calendar fragments or unstable volume spikes, that would be much less convincing.


## Error Analysis

This is one of the most overlooked sections in public notebooks.

A model can look strong on average and still fail exactly where the market is hardest. So we test whether errors expand in turbulent conditions.


In [ ]:
# 16) Error analysis by volatility regime
error_df = ready.iloc[split_idx:].copy()
error_df['pred_baseline'] = pred_baseline
error_df['pred_improved'] = pred_improved
error_df['abs_err_baseline'] = (error_df['target_next_close'] - error_df['pred_baseline']).abs()
error_df['abs_err_improved'] = (error_df['target_next_close'] - error_df['pred_improved']).abs()
error_df['vol_quartile'] = pd.qcut(error_df['Rolling_Vol_21'], 4, labels=['Q1 Calm', 'Q2', 'Q3', 'Q4 Turbulent'])

error_summary = error_df.groupby('vol_quartile')[['abs_err_baseline', 'abs_err_improved']].mean().round(3)
display(error_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=error_df, x='vol_quartile', y='abs_err_baseline', color=PALETTE[1], ax=axes[0])
axes[0].set_title('Baseline absolute error by volatility quartile')
axes[0].set_xlabel('')
axes[0].set_ylabel('Absolute error')

sns.barplot(data=error_df, x='vol_quartile', y='abs_err_improved', color=PALETTE[0], ax=axes[1])
axes[1].set_title('Improved model absolute error by volatility quartile')
axes[1].set_xlabel('')
axes[1].set_ylabel('Absolute error')

plt.tight_layout()
plt.show()


## Final Insights

The notebook now tells a coherent story from raw data to interpretable model behavior.

The strongest practical takeaways are:
- gold behaves in regimes, not as one stationary process,
- volatility clustering is real, but high volatility is not automatically bearish,
- seasonality is weak but persistent enough to deserve feature space,
- a nonlinear model improves on the linear baseline because regime interactions matter.


## What Worked / What Didn’t

### What worked
- Lagged closes and rolling trend features provided a strong baseline signal.
- Regime-aware variables made the improved model more adaptive.
- Holdout validation and error slices made the notebook more trustworthy than a CV-only report.

### What didn’t
- Raw volume is noisy and partially inconsistent because some rows contain zero volume.
- Short-horizon forecasting still struggles most in turbulent windows.
- Even the improved model is not a trading strategy on its own; it is a forecasting baseline.


## Pitfalls Section

A few mistakes are easy to make on this dataset:
- using random train/test splits instead of time-aware validation,
- leaking future information through centered rolling windows,
- treating zero volume as ordinary activity without checking its prevalence,
- optimizing only for headline score and skipping regime-specific failures.


## Next Steps

If you want to push this notebook beyond a strong public baseline, the best next moves are:
1. Predict next-day return or direction as a tougher target alongside next-day close.
2. Add external macro drivers such as USD index, rates, or inflation proxies.
3. Test walk-forward retraining instead of a single terminal holdout.
4. Compare against LightGBM or CatBoost if the competition rules allow them.
5. Add probabilistic intervals, not just point forecasts.


## Conclusion

This notebook focused on being useful rather than noisy. We extracted several high-signal views of the gold market, built a clean forecasting pipeline, and showed where the model helps and where it still struggles.

For a public Kaggle notebook, that combination matters: readers want insight, rigor, and something they can reuse in their own work.

If you found this useful, feel free to upvote 👍
